# Movie Review Sentiment Classification with BERT
**Portfolio presentation: Cameron Batts**

Collaborative project. Individual contributions are not specified in the supplied notebook; this presentation does not claim sole authorship.

Python · NLP · Transformers · PyTorch · Naive Bayes · Logistic Regression

## Overview
Compare bag-of-words classifiers with a fully fine-tuned BERT model for binary sentiment classification of Rotten Tomatoes reviews. Preserve the dataset's predefined training, validation and test splits and compare all models on the same 1,066-review test set.

## Business context
Review classification can support content triage and aggregate audience-feedback analysis. This study measures benchmark classification performance rather than a deployed business outcome.

## Data and source identifiers
Hugging Face dataset: `cornell-movie-review-data/rotten_tomatoes`, https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes . Model checkpoint: `MarieAngeA13/Sentiment-Analysis-BERT`. These identifiers are retained for attribution and reproducibility. Raw reviews and person-name examples are not bundled.

## Execution status
Selected numerical outputs are retained from the supplied team notebook. The cleaned code is syntax-checked but not rerun; trained weights are not included. Performance is from one reported run, not independent reproduction.

In [ ]:
# General dataframe imports
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)

# sklearn imports
from sklearn.metrics import classification_report, accuracy_score
from sklearn import naive_bayes
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Hugging Face/BERT imports
from datasets import load_dataset
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

## Dataset splits
Load the source dataset at runtime. It provides 8,530 training, 1,066 validation and 1,066 test reviews. Training labels are balanced: 4,265 per class; test support is 533 per class. Labels are 0=negative and 1=positive. Dataset and model downloads require internet access.

In [ ]:
# Download the dataset
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

# Store each split as a pandas dataframe
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Training observations:", len(train_df))
print("Validation observations:", len(val_df))
print("Testing observations:", len(test_df))

In [ ]:
for frame in [train_df, val_df, test_df]:
    frame['sentiment'] = frame['label'].map({0: 'negative', 1: 'positive'})
print(train_df['sentiment'].value_counts())


In [ ]:
# Set up X and y for each split
X_train = train_df["text"]
y_train = train_df["label"]

X_val = val_df["text"]
y_val = val_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (8530,)
X_val: (1066,)
X_test: (1066,)


## Bag-of-words representation
Use unigrams and bigrams with English stopword removal. Fit the vocabulary on training text only and reuse it for validation and test transformations. The original vocabulary contains 83,753 features. Stopword removal can suppress sentiment cues such as negation, so its effect deserves validation.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words="english")

# Fit only on the training data
X_train_vectors = vectorizer.fit_transform(X_train)

# Transform validation and test data using the training vocabulary
X_val_vectors = vectorizer.transform(X_val)
X_test_vectors = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Vocabulary size: 83753


## Naive Bayes baseline

In [ ]:
# Train the Naive Bayes model
model_nb = naive_bayes.MultinomialNB()
model_nb.fit(X_train_vectors, y_train)

# Predict the testing data
y_pred_nb = model_nb.predict(X_test_vectors)

# Display performance
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0.0
))

              precision    recall  f1-score   support

    negative     0.7761    0.7805    0.7783       533
    positive     0.7792    0.7749    0.7770       533

    accuracy                         0.7777      1066
   macro avg     0.7777    0.7777    0.7777      1066
weighted avg     0.7777    0.7777    0.7777      1066



## Logistic regression baseline

In [ ]:
# Train the Logistic Regression model
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_vectors, y_train)

# Predict the testing data
y_pred_lr = model_lr.predict(X_test_vectors)

# Display performance
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0.0
))

              precision    recall  f1-score   support

    negative     0.7646    0.7617    0.7632       533
    positive     0.7626    0.7655    0.7640       533

    accuracy                         0.7636      1066
   macro avg     0.7636    0.7636    0.7636      1066
weighted avg     0.7636    0.7636    0.7636      1066



## BERT setup
Use the checkpoint and tokenizer from the original analysis. CUDA is selected when available, otherwise CPU. The reported run used a Tesla T4 GPU; runtime is hardware-dependent.

In [ ]:
# BERT model and tokenizer
bert_path = "MarieAngeA13/Sentiment-Analysis-BERT"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_path)

# Check whether a GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Function to tokenize and convert the data to tensors
def tokenize_and_tensorize(data, labels, tokenizer):
    tokens = tokenizer(
        data.tolist(),
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    labels_tensor = torch.tensor(labels.tolist())

    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"]
    }, labels_tensor


train_encodings, y_train_tensor = tokenize_and_tensorize(
    X_train, y_train, bert_tokenizer
)

val_encodings, y_val_tensor = tokenize_and_tensorize(
    X_val, y_val, bert_tokenizer
)

test_encodings, y_test_tensor = tokenize_and_tensorize(
    X_test, y_test, bert_tokenizer
)

In [ ]:
class RottenTomatoesDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: val[idx].clone().detach()
            for key, val in self.encodings.items()
        }

        item["labels"] = self.labels[idx].clone().detach()
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = RottenTomatoesDataset(train_encodings, y_train_tensor)
val_dataset = RottenTomatoesDataset(val_encodings, y_val_tensor)
test_dataset = RottenTomatoesDataset(test_encodings, y_test_tensor)

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

## Binary classification head and training
The source checkpoint has three output classes. Requesting two labels with `ignore_mismatched_sizes=True` reinitializes the incompatible classifier weights and bias while retaining compatible encoder weights. This mismatch is an intentional task adaptation; the new head must be trained.

Fine-tune all parameters for two epochs using learning_rate=2e-5, weight_decay=0.01 and seed=42. Training batch size is 16 and evaluation batch size is 32. Save and evaluate each epoch, then select the checkpoint with lowest validation loss. CUDA runs enable fp16.

In [ ]:
# Load BERT for binary sequence classification
bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_path,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
    ignore_mismatched_sizes=True
)

# Training settings
training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
    seed=42,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# Fine-tune BERT
trainer.train()

## Held-out evaluation
Evaluate the selected BERT checkpoint on the same predefined test split as the baselines.

In [ ]:
# Predict the test data
bert_output = trainer.predict(test_dataset)

# Convert the model output into predicted labels
y_pred_bert = np.argmax(bert_output.predictions, axis=1)

# Display performance
print(classification_report(
    y_test_tensor.numpy(),
    y_pred_bert,
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0.0
))

              precision    recall  f1-score   support

    negative     0.8857    0.7992    0.8402       533
    positive     0.8171    0.8968    0.8551       533

    accuracy                         0.8480      1066
   macro avg     0.8514    0.8480    0.8477      1066
weighted avg     0.8514    0.8480    0.8477      1066



In [ ]:
# Calculate accuracy for each model
nb_accuracy = accuracy_score(y_test, y_pred_nb)
lr_accuracy = accuracy_score(y_test, y_pred_lr)
bert_accuracy = accuracy_score(y_test_tensor.numpy(), y_pred_bert)

results_df = pd.DataFrame({
    "Model": [
        "Naive Bayes",
        "Logistic Regression",
        "Fine-tuned BERT"
    ],
    "Accuracy": [
        nb_accuracy,
        lr_accuracy,
        bert_accuracy
    ]
})

results_df

                 Model  Accuracy
0          Naive Bayes  0.777674
1  Logistic Regression  0.763602
2      Fine-tuned BERT  0.848030

## Results and interpretation
Fine-tuned BERT reported accuracy 0.848030 (84.80%) and macro F1 0.8477. Naive Bayes reached 0.777674 accuracy and 0.7777 macro F1; logistic regression reached 0.763602 accuracy and 0.7636 macro F1. BERT's accuracy gain was approximately 7.04 percentage points over Naive Bayes and 8.44 points over logistic regression.

BERT's positive-class recall was 0.8968 and negative-class recall was 0.7992, showing an asymmetric error pattern despite balanced test support. The result supports BERT for this experiment, not a universal algorithm ranking.

## Limitations and next steps
- The cleaned workflow was not rerun; dataset/model revisions and package versions are unpinned.
- Only one training seed and dataset split were reported. No confidence interval or significance test was computed.
- Prior checkpoint exposure to related review data has not been audited; independent performance on new reviews is unknown.
- Compare negation-preserving vectorization and tune baseline regularization on validation data.
- Audit errors, evaluate multiple seeds, and compare compute/memory requirements before deployment.

## Contribution context
This is a team-developed analysis presented in Cameron Batts's portfolio. The supplied notebook does not identify individual responsibilities. Shared results should be described as collaborative rather than solely authored.